# Walmart - Photo Center: Personalized Gift Performance Analysis

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [2]:
df_photo = pd.read_csv('../Data/011/photo_gift_sales_2.csv', parse_dates=['purchase_date'])

pl_photo = pl.read_csv('../Data/011/photo_gift_sales_2.csv', try_parse_dates=True)

# Pregunta 1

### Para cada producto de regalo fotográfico personalizado, ¿cuál es la cantidad total comprada en abril de 2024? Este resultado proporcionará una medida clara del rendimiento del producto para nuestras estrategias de inventario.

```SQL
SELECT
    product_id,
    SUM(quantity) AS total_quantity
FROM fct_photo_gift_sales
WHERE ((EXTRACT(MONTH FROM purchase_date) = 4) AND
       (EXTRACT(YEAR FROM purchase_date) = 2024))
GROUP BY product_id;
```

In [4]:
abril = df_photo[
    (df_photo['purchase_date'].dt.month == 4) &
    (df_photo['purchase_date'].dt.year == 2024)
].reset_index()

res = abril.groupby('product_id').agg(
    total_quantity = ('quantity','sum')
).reset_index()

res = res[['product_id','total_quantity']]

In [6]:
res = pl_photo.filter(
    (pl.col('purchase_date').dt.month() == 4) &
    (pl.col('purchase_date').dt.year() == 2024)
).group_by('product_id').agg(
    pl.col('quantity').sum().alias('total_quantity')
)

# Pregunta 2

### ¿Cuál es la cantidad máxima de regalos fotográficos personalizados comprados en una sola transacción durante abril de 2024? Esta información resaltará el comportamiento de compra máximo para transacciones individuales.

```SQL
SELECT
    MAX(quantity)
FROM fct_photo_gift_sales
WHERE ((EXTRACT(MONTH FROM purchase_date) = 4) AND
       (EXTRACT(YEAR FROM purchase_date) = 2024));
```

In [13]:
abril = df_photo[
    (df_photo['purchase_date'].dt.month == 4) &
    (df_photo['purchase_date'].dt.year == 2024)
]['quantity'].max()


In [14]:
res = pl_photo.filter(
    (pl.col('purchase_date').dt.month() == 4) &
    (pl.col('purchase_date').dt.year() == 2024)
).select(
    pl.col('quantity').max().alias('max_quantity')
)

# Pregunta 3

### ¿Cuál es el promedio general de regalos fotográficos personalizados comprados por cliente durante abril de 2024? Es decir, para cada cliente, calcula el número total de regalos que compró en abril de 2024; luego, devuelve el promedio de esos valores entre todos los clientes.

```SQL
WITH total_regalos AS(
    SELECT
        customer_id,
        SUM(quantity) AS sum_regalos
    FROM fct_photo_gift_sales
    WHERE ((EXTRACT(MONTH FROM purchase_date) = 4) AND
           (EXTRACT(YEAR FROM purchase_date) = 2024))
    GROUP BY customer_id
)
SELECT
    ROUND(AVG(sum_regalos),2) AS avg_regalos
FROM total_regalos
```

In [18]:
abril = df_photo[
    (df_photo['purchase_date'].dt.month == 4) &
    (df_photo['purchase_date'].dt.year == 2024)
].reset_index()

res = abril.groupby('customer_id').agg(
    sum_regalos = ('quantity', 'sum')
).reset_index()

res = res[['customer_id','sum_regalos']]

res_avg = res.agg(
    avg_regalos = ('sum_regalos', 'mean')
).round(2)

res_avg

,sum_regalos
avg_regalos,4.83


In [22]:
res = pl_photo.filter(
    (pl.col('purchase_date').dt.month() == 4) &
    (pl.col('purchase_date').dt.year() == 2024)
).group_by('customer_id').agg(
    pl.col('quantity').sum().alias('sum_regalos')
)

res_avg = res.select(
    pl.col('sum_regalos').mean().round(2).alias('avg_regalos')
)

res_avg

avg_regalos
f64
4.83
